# Notebook 06 — Repurchase Prediction

## Objetivo
Construir um modelo **out-of-time** para estimar a probabilidade de um cliente realizar uma segunda compra em até **90 dias** após sua primeira compra entregue.

O notebook prioriza rigor analítico e utilidade de negócio:
- evita **data leakage** usando apenas informações disponíveis na primeira compra;
- usa **split temporal** em treino, validação e teste;
- trata o desbalanceamento com métricas adequadas a eventos raros (**PR-AUC, lift, precision@k e recall@k**), em vez de depender de acurácia;
- compara **Logistic Regression** e **Random Forest** contra um baseline ingênuo;
- converte os scores em faixas de priorização para campanhas de retenção;
- avalia estabilidade temporal e limitações do modelo.

> **Target:** `repurchase_90d = 1` quando a segunda compra entregue acontece até 90 dias após a primeira compra entregue.

## 1. Configuração e bibliotecas

In [ ]:
from pathlib import Path
import warnings
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.base import clone

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

RANDOM_STATE = 42
PROJECT_ROOT = Path('/mnt/data/olist_project') if Path('/mnt/data/olist_project').exists() else Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
FALLBACK_RAW_DIR = Path('/mnt/data/olist_data')
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('MODEL_DIR:', MODEL_DIR)

## 2. Carregamento/reconstrução da base de modelagem

O Notebook 04 deveria exportar `repurchase_90d_model_base.csv`. Para tornar este notebook **reprodutível**, a função abaixo reconstrói essa base caso o arquivo não exista ou esteja incompleto.

A reconstrução usa:
- `orders_base.csv` do Notebook 01;
- pagamentos e itens brutos;
- produtos e tradução de categorias.

Nenhuma informação posterior à primeira compra entra nas features.

In [ ]:
def raw_path(filename):
    p = RAW_DIR / filename
    if p.exists():
        return p
    fallback = FALLBACK_RAW_DIR / filename
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f'Arquivo não encontrado: {filename}')


def build_repurchase_model_base():
    orders_base = pd.read_csv(
        PROCESSED_DIR / 'orders_base.csv',
        parse_dates=['order_purchase_timestamp']
    )
    payments = pd.read_csv(raw_path('olist_order_payments_dataset.csv'))
    order_items = pd.read_csv(raw_path('olist_order_items_dataset.csv'))
    products = pd.read_csv(raw_path('olist_products_dataset.csv'))
    category_translation = pd.read_csv(raw_path('product_category_name_translation.csv'))

    delivered = orders_base.loc[
        (orders_base['order_status'] == 'delivered') &
        orders_base['customer_unique_id'].notna() &
        orders_base['order_purchase_timestamp'].notna()
    ].copy()
    delivered = delivered.sort_values(
        ['customer_unique_id', 'order_purchase_timestamp', 'order_id']
    ).reset_index(drop=True)
    delivered['order_number'] = delivered.groupby('customer_unique_id').cumcount() + 1
    dataset_end = delivered['order_purchase_timestamp'].max()

    first_orders = delivered.loc[delivered['order_number'] == 1].copy()
    second_orders = delivered.loc[
        delivered['order_number'] == 2,
        ['customer_unique_id', 'order_id', 'order_purchase_timestamp']
    ].rename(columns={
        'order_id': 'second_order_id',
        'order_purchase_timestamp': 'second_purchase'
    })

    customer = first_orders.merge(
        second_orders, on='customer_unique_id', how='left', validate='one_to_one'
    ).rename(columns={
        'order_id': 'first_order_id',
        'order_purchase_timestamp': 'first_purchase',
        'payment_value': 'first_order_payment_value',
        'product_value': 'first_order_product_value',
        'freight_value': 'first_order_freight_value',
        'item_count': 'first_order_item_count'
    })

    customer['eligible_90d'] = customer['first_purchase'] <= dataset_end - pd.Timedelta(days=90)
    customer['repurchase_90d'] = (
        customer['second_purchase'].notna() &
        (customer['second_purchase'] <= customer['first_purchase'] + pd.Timedelta(days=90))
    )

    item_features = (
        order_items.groupby('order_id', as_index=False)
        .agg(
            first_order_distinct_products=('product_id', 'nunique'),
            first_order_distinct_sellers=('seller_id', 'nunique')
        )
        .rename(columns={'order_id': 'first_order_id'})
    )
    customer = customer.merge(item_features, on='first_order_id', how='left', validate='one_to_one')

    payment_type_value = (
        payments.groupby(['order_id', 'payment_type'], as_index=False)['payment_value'].sum()
    )
    primary_idx = payment_type_value.groupby('order_id')['payment_value'].idxmax()
    primary_payment = payment_type_value.loc[
        primary_idx, ['order_id', 'payment_type']
    ].rename(columns={'payment_type': 'primary_payment_type'})

    payment_features = (
        payments.groupby('order_id', as_index=False)
        .agg(
            max_installments=('payment_installments', 'max'),
            payment_record_count=('payment_sequential', 'count'),
            payment_type_count=('payment_type', 'nunique')
        )
        .merge(primary_payment, on='order_id', how='left', validate='one_to_one')
        .rename(columns={'order_id': 'first_order_id'})
    )
    customer = customer.merge(payment_features, on='first_order_id', how='left', validate='one_to_one')

    product_category = products.merge(category_translation, on='product_category_name', how='left')
    product_category['category'] = (
        product_category['product_category_name_english']
        .fillna(product_category['product_category_name'])
        .fillna('unknown')
    )
    items_enriched = order_items.merge(
        product_category[['product_id', 'category']],
        on='product_id', how='left', validate='many_to_one'
    )
    items_enriched['category'] = items_enriched['category'].fillna('unknown')
    order_category_value = (
        items_enriched.groupby(['order_id', 'category'], as_index=False)['price'].sum()
    )
    category_idx = order_category_value.groupby('order_id')['price'].idxmax()
    dominant_category = order_category_value.loc[
        category_idx, ['order_id', 'category']
    ].rename(columns={'order_id': 'first_order_id', 'category': 'dominant_category'})
    customer = customer.merge(dominant_category, on='first_order_id', how='left', validate='one_to_one')

    customer['dominant_category'] = customer['dominant_category'].fillna('unknown')
    customer['freight_ratio'] = np.where(
        customer['first_order_payment_value'] > 0,
        customer['first_order_freight_value'] / customer['first_order_payment_value'],
        np.nan
    )
    customer['first_purchase_month'] = customer['first_purchase'].dt.to_period('M').astype(str)
    customer['first_purchase_month_of_year'] = customer['first_purchase'].dt.month
    customer['first_purchase_dow'] = customer['first_purchase'].dt.day_name()
    customer['first_purchase_hour'] = customer['first_purchase'].dt.hour

    columns = [
        'customer_unique_id', 'first_order_id', 'first_purchase',
        'first_order_payment_value', 'first_order_product_value',
        'first_order_freight_value', 'first_order_item_count',
        'first_order_distinct_products', 'first_order_distinct_sellers',
        'freight_ratio', 'primary_payment_type', 'max_installments',
        'payment_record_count', 'payment_type_count', 'dominant_category',
        'first_purchase_month', 'first_purchase_month_of_year',
        'first_purchase_dow', 'first_purchase_hour', 'repurchase_90d'
    ]

    model_base = (
        customer.loc[customer['eligible_90d'], columns]
        .copy()
        .sort_values('first_purchase')
        .reset_index(drop=True)
    )
    return model_base


model_path = PROCESSED_DIR / 'repurchase_90d_model_base.csv'
required_columns = {
    'first_order_distinct_products', 'first_order_distinct_sellers',
    'first_purchase_month_of_year', 'repurchase_90d'
}

rebuild = True
if model_path.exists():
    probe = pd.read_csv(model_path, nrows=5)
    rebuild = not required_columns.issubset(probe.columns)

if rebuild:
    print('Reconstruindo base de modelagem...')
    model_base = build_repurchase_model_base()
    model_base.to_csv(model_path, index=False)
else:
    model_base = pd.read_csv(model_path, parse_dates=['first_purchase'])

if not pd.api.types.is_datetime64_any_dtype(model_base['first_purchase']):
    model_base['first_purchase'] = pd.to_datetime(model_base['first_purchase'])

model_base['repurchase_90d'] = model_base['repurchase_90d'].astype(int)
print('Model base:', model_base.shape)
print('Período:', model_base['first_purchase'].min(), '→', model_base['first_purchase'].max())
print('Target rate:', f"{model_base['repurchase_90d'].mean():.2%}")
display(model_base.head())

## 3. Sanity checks e perfil do target

A recompra em 90 dias é um **evento raro**. Nesse cenário, acurácia é pouco informativa: um modelo que prevê sempre “não recompra” teria acurácia muito alta, mas nenhuma utilidade comercial.

In [ ]:
assert model_base['customer_unique_id'].is_unique, 'Deve haver uma linha por cliente.'
assert model_base['repurchase_90d'].isin([0, 1]).all(), 'Target inválido.'
assert model_base['first_purchase'].is_monotonic_increasing, 'Base deve estar ordenada no tempo.'
assert model_base['first_purchase'].notna().all(), 'Data da primeira compra não pode ser nula.'

class_summary = (
    model_base['repurchase_90d']
    .value_counts()
    .rename_axis('repurchase_90d')
    .reset_index(name='customers')
)
class_summary['share'] = class_summary['customers'] / class_summary['customers'].sum()
display(class_summary)

plt.figure(figsize=(6, 4))
plt.bar(class_summary['repurchase_90d'].astype(str), class_summary['share'] * 100)
plt.title('Distribuição do target — recompra em 90 dias')
plt.xlabel('repurchase_90d')
plt.ylabel('Clientes (%)')
plt.tight_layout()
plt.show()

## 4. Definição das features

### Features numéricas
São atributos da primeira compra: ticket, valor de produtos, frete, quantidade de itens/produtos/vendedores, razão de frete, parcelamento e complexidade do pagamento.

### Features categóricas
- meio de pagamento principal;
- categoria dominante;
- dia da semana.

O mês absoluto de aquisição (`YYYY-MM`) **não entra como feature**, pois poderia fazer o modelo decorar tendência temporal. Mantemos apenas o mês do ano (1–12) para capturar sazonalidade.

In [ ]:
numeric_features = [
    'first_order_payment_value',
    'first_order_product_value',
    'first_order_freight_value',
    'first_order_item_count',
    'first_order_distinct_products',
    'first_order_distinct_sellers',
    'freight_ratio',
    'max_installments',
    'payment_record_count',
    'payment_type_count',
    'first_purchase_month_of_year',
    'first_purchase_hour'
]

categorical_features = [
    'primary_payment_type',
    'dominant_category',
    'first_purchase_dow'
]

feature_columns = numeric_features + categorical_features

display(model_base[feature_columns].isna().sum().to_frame('nulls').sort_values('nulls', ascending=False))

## 5. Split temporal: treino, validação e teste

Usamos **70% / 15% / 15% em ordem cronológica**:
- treino: ajuste dos modelos;
- validação: comparação e seleção;
- teste: avaliação final out-of-time.

Isso simula o cenário real: treinar com passado e prever clientes adquiridos posteriormente.

In [ ]:
n = len(model_base)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_df = model_base.iloc[:train_end].copy()
valid_df = model_base.iloc[train_end:valid_end].copy()
test_df = model_base.iloc[valid_end:].copy()

split_summary = pd.DataFrame([
    ['train', len(train_df), train_df['first_purchase'].min(), train_df['first_purchase'].max(), train_df['repurchase_90d'].mean()],
    ['validation', len(valid_df), valid_df['first_purchase'].min(), valid_df['first_purchase'].max(), valid_df['repurchase_90d'].mean()],
    ['test', len(test_df), test_df['first_purchase'].min(), test_df['first_purchase'].max(), test_df['repurchase_90d'].mean()],
], columns=['split', 'customers', 'start', 'end', 'repurchase_rate_90d'])
display(split_summary)

## 6. Pré-processamento

- numéricas: imputação pela mediana + padronização;
- categóricas: imputação pela moda + one-hot encoding;
- categorias muito raras são agrupadas via `min_frequency=50`.

O pipeline é ajustado **somente no treino**, evitando vazamento de informação.

In [ ]:
def make_preprocessor():
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50))
    ])
    return ColumnTransformer([
        ('num', numeric_pipe, numeric_features),
        ('cat', categorical_pipe, categorical_features)
    ])

X_train = train_df[feature_columns]
y_train = train_df['repurchase_90d']
X_valid = valid_df[feature_columns]
y_valid = valid_df['repurchase_90d']
X_test = test_df[feature_columns]
y_test = test_df['repurchase_90d']

## 7. Métricas para evento raro

A métrica principal é **PR-AUC (Average Precision)**, porque ela se concentra no desempenho da classe positiva em bases desbalanceadas.

Também calculamos:
- **ROC-AUC** — capacidade geral de ordenação;
- **Brier Score** — qualidade probabilística (menor é melhor);
- **Precision@K** — taxa de recompra entre os K% clientes mais bem ranqueados;
- **Recall@K** — proporção dos repurchasers capturada nos K% superiores;
- **Lift@K** — quantas vezes a taxa do grupo priorizado supera a taxa média.

In [ ]:
def top_k_metrics(y_true, y_score, k):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n_top = max(1, int(len(y_true) * k))
    idx = np.argsort(y_score)[::-1][:n_top]
    prevalence = y_true.mean()
    precision_k = y_true[idx].mean()
    recall_k = y_true[idx].sum() / y_true.sum() if y_true.sum() else np.nan
    lift_k = precision_k / prevalence if prevalence else np.nan
    return precision_k, recall_k, lift_k


def evaluate_scores(y_true, y_score, model_name, split_name):
    row = {
        'model': model_name,
        'split': split_name,
        'prevalence': np.mean(y_true),
        'pr_auc': average_precision_score(y_true, y_score),
        'roc_auc': roc_auc_score(y_true, y_score),
        'brier_score': brier_score_loss(y_true, y_score),
    }
    for k in [0.05, 0.10, 0.20]:
        precision_k, recall_k, lift_k = top_k_metrics(y_true, y_score, k)
        label = int(k * 100)
        row[f'precision_at_{label}pct'] = precision_k
        row[f'recall_at_{label}pct'] = recall_k
        row[f'lift_at_{label}pct'] = lift_k
    return row

## 8. Baseline e modelos candidatos

Comparamos:
1. **Dummy Prior** — sempre devolve a prevalência observada no treino;
2. **Logistic Regression** — benchmark simples e interpretável;
3. **Random Forest** — captura interações e não linearidades.

Não usamos oversampling aleatório, pois ele pode distorcer a estrutura temporal e a calibração. O desbalanceamento é tratado principalmente pela **métrica de seleção e pela estratégia de ranking top-K**.

In [ ]:
candidates = {
    'Dummy Prior': DummyClassifier(strategy='prior'),
    'Logistic Regression': LogisticRegression(
        C=0.5,
        max_iter=1200,
        solver='liblinear',
        random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=250,
        max_depth=12,
        min_samples_leaf=20,
        max_features='sqrt',
        n_jobs=2,
        random_state=RANDOM_STATE
    )
}

validation_rows = []
validation_scores = {}

for name, estimator in candidates.items():
    pipe = Pipeline([
        ('preprocessor', make_preprocessor()),
        ('model', estimator)
    ])
    pipe.fit(X_train, y_train)
    score = pipe.predict_proba(X_valid)[:, 1]
    validation_scores[name] = score
    validation_rows.append(evaluate_scores(y_valid, score, name, 'validation'))

validation_metrics = pd.DataFrame(validation_rows).sort_values('pr_auc', ascending=False)
display(validation_metrics)

## 9. Curvas de Precision–Recall e ROC na validação

In [ ]:
plt.figure(figsize=(8, 5))
for name, score in validation_scores.items():
    precision, recall, _ = precision_recall_curve(y_valid, score)
    plt.plot(recall, precision, label=name)
plt.axhline(y_valid.mean(), linestyle='--', label='Prevalência')
plt.title('Precision–Recall — validação temporal')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
for name, score in validation_scores.items():
    fpr, tpr, _ = roc_curve(y_valid, score)
    plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.title('ROC — validação temporal')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Seleção do modelo

Selecionamos o melhor candidato por **PR-AUC na validação**, excluindo o baseline Dummy. O objetivo principal é ordenar melhor os poucos clientes que efetivamente recompram.

In [ ]:
eligible_models = validation_metrics.loc[validation_metrics['model'] != 'Dummy Prior'].copy()
best_model_name = eligible_models.iloc[0]['model']
print('Modelo selecionado:', best_model_name)
print('PR-AUC validação:', f"{eligible_models.iloc[0]['pr_auc']:.4f}")
print('Lift@10% validação:', f"{eligible_models.iloc[0]['lift_at_10pct']:.2f}x")

## 11. Refit em treino + validação e teste final out-of-time

Após escolher o algoritmo, fazemos o refit com os primeiros **85%** dos clientes e avaliamos uma única vez nos **15% mais recentes**.

In [ ]:
train_valid_df = model_base.iloc[:valid_end].copy()
X_train_valid = train_valid_df[feature_columns]
y_train_valid = train_valid_df['repurchase_90d']

best_estimator = clone(candidates[best_model_name])
final_model = Pipeline([
    ('preprocessor', make_preprocessor()),
    ('model', best_estimator)
])
final_model.fit(X_train_valid, y_train_valid)

test_score = final_model.predict_proba(X_test)[:, 1]
test_metrics = pd.DataFrame([
    evaluate_scores(y_test, test_score, best_model_name, 'test')
])
display(test_metrics)

## 12. Comparação validação × teste: estabilidade temporal

Queda de performance no teste é um sinal de **drift temporal**. Em produção, isso justificaria monitoramento periódico, retreino e revisão de features.

In [ ]:
stability = pd.concat([
    validation_metrics.loc[validation_metrics['model'] == best_model_name],
    test_metrics
], ignore_index=True)
display(stability[['model', 'split', 'prevalence', 'pr_auc', 'roc_auc', 'brier_score', 'lift_at_10pct']])

plt.figure(figsize=(7, 4))
plt.bar(stability['split'], stability['pr_auc'])
plt.title(f'PR-AUC — estabilidade temporal ({best_model_name})')
plt.ylabel('PR-AUC')
plt.tight_layout()
plt.show()

## 13. Curva de calibração

A calibração verifica se grupos com score maior realmente apresentam maior frequência de recompra. Para campanhas, o ranking é o uso principal; ainda assim, boa calibração é necessária caso o score seja interpretado literalmente como probabilidade.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, test_score, n_bins=8, strategy='quantile')
plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker='o', label=best_model_name)
plt.plot([0, max(prob_pred.max(), prob_true.max())], [0, max(prob_pred.max(), prob_true.max())], linestyle='--', label='Calibração perfeita')
plt.title('Curva de calibração — teste temporal')
plt.xlabel('Probabilidade média prevista')
plt.ylabel('Frequência observada')
plt.legend()
plt.tight_layout()
plt.show()

calibration_table = pd.DataFrame({
    'mean_predicted_probability': prob_pred,
    'observed_repurchase_rate': prob_true
})
display(calibration_table)

## 14. Priorização por decis de score

Em vez de usar um threshold fixo de 0,5 — inadequado para um target de ~2% — ordenamos os clientes pelo score e dividimos em **10 decis**.

O **Decil 1** contém os 10% com maior score de recompra.

In [ ]:
scored_test = test_df[[
    'customer_unique_id', 'first_order_id', 'first_purchase', 'repurchase_90d',
    'first_order_payment_value', 'primary_payment_type', 'dominant_category'
]].copy()
scored_test['repurchase_score'] = test_score
scored_test['score_decile'] = (
    pd.qcut(
        scored_test['repurchase_score'].rank(method='first', ascending=False),
        q=10,
        labels=False
    ) + 1
)

baseline_rate = scored_test['repurchase_90d'].mean()
decile_summary = (
    scored_test.groupby('score_decile')
    .agg(
        customers=('customer_unique_id', 'size'),
        repurchasers=('repurchase_90d', 'sum'),
        observed_repurchase_rate=('repurchase_90d', 'mean'),
        avg_score=('repurchase_score', 'mean'),
        avg_first_ticket=('first_order_payment_value', 'mean')
    )
    .reset_index()
    .sort_values('score_decile')
)
decile_summary['lift_vs_average'] = decile_summary['observed_repurchase_rate'] / baseline_rate
display(decile_summary)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(decile_summary['score_decile'].astype(str), decile_summary['observed_repurchase_rate'] * 100)
plt.axhline(baseline_rate * 100, linestyle='--', label='Média do teste')
plt.title('Taxa observada de recompra por decil de score')
plt.xlabel('Decil de score — 1 = maior propensão')
plt.ylabel('Recompra observada em 90 dias (%)')
plt.legend()
plt.tight_layout()
plt.show()

## 15. Simulação de campanha por orçamento

A tabela responde a uma pergunta executiva: **se só pudermos contatar 5%, 10% ou 20% da base, quantos repurchasers conseguimos concentrar?**

O notebook não calcula ROI financeiro porque ainda não temos custo de contato, margem incremental nem uplift causal.

In [ ]:
campaign_rows = []
for k in [0.05, 0.10, 0.20, 0.30]:
    p, r, l = top_k_metrics(y_test, test_score, k)
    n_target = int(len(test_df) * k)
    campaign_rows.append({
        'target_share': k,
        'target_customers': n_target,
        'precision_in_target': p,
        'recall_of_repurchasers': r,
        'lift_vs_random': l,
        'expected_repurchasers_observed': int(round(p * n_target))
    })

campaign_summary = pd.DataFrame(campaign_rows)
display(campaign_summary)

## 16. Importância das features

Para Random Forest usamos `feature_importances_`. Para Logistic Regression usamos o valor absoluto dos coeficientes padronizados. Esses resultados são **associativos**, não causais.

In [ ]:
preprocessor = final_model.named_steps['preprocessor']
model = final_model.named_steps['model']
feature_names = preprocessor.get_feature_names_out()

if hasattr(model, 'feature_importances_'):
    importance_values = model.feature_importances_
    importance_type = 'feature_importance'
elif hasattr(model, 'coef_'):
    importance_values = np.abs(model.coef_[0])
    importance_type = 'abs_coefficient'
else:
    importance_values = np.full(len(feature_names), np.nan)
    importance_type = 'importance'

feature_importance = pd.DataFrame({
    'feature': feature_names,
    importance_type: importance_values
}).sort_values(importance_type, ascending=False).reset_index(drop=True)

display(feature_importance.head(25))

In [ ]:
top_imp = feature_importance.head(20).sort_values(importance_type)
plt.figure(figsize=(10, 7))
plt.barh(top_imp['feature'], top_imp[importance_type])
plt.title(f'Top 20 features — {best_model_name}')
plt.xlabel(importance_type)
plt.ylabel('Feature transformada')
plt.tight_layout()
plt.show()

## 17. Performance por mês no período de teste

Essa visão ajuda a detectar instabilidade ao longo do tempo. PR-AUC só é calculada quando há classes positiva e negativa no mês.

In [ ]:
monthly_test = scored_test.copy()
monthly_test['first_purchase_month'] = monthly_test['first_purchase'].dt.to_period('M').astype(str)

monthly_rows = []
for month, g in monthly_test.groupby('first_purchase_month'):
    row = {
        'first_purchase_month': month,
        'customers': len(g),
        'repurchasers': int(g['repurchase_90d'].sum()),
        'repurchase_rate': g['repurchase_90d'].mean(),
        'avg_score': g['repurchase_score'].mean()
    }
    if g['repurchase_90d'].nunique() == 2:
        row['pr_auc'] = average_precision_score(g['repurchase_90d'], g['repurchase_score'])
        row['roc_auc'] = roc_auc_score(g['repurchase_90d'], g['repurchase_score'])
    else:
        row['pr_auc'] = np.nan
        row['roc_auc'] = np.nan
    monthly_rows.append(row)

monthly_performance = pd.DataFrame(monthly_rows)
display(monthly_performance)

## 18. Interpretação executiva

### Como usar o modelo
O score deve ser usado como **ranking de priorização**, e não como certeza individual de recompra. Em uma operação real, os clientes do topo do ranking poderiam receber:
- incentivo para segunda compra;
- cross-sell baseado na categoria inicial;
- comunicação dentro da janela de 30–90 dias identificada no Notebook 04;
- benefício diferenciado conforme ticket e comportamento de pagamento.

### O que o modelo não prova
- Não mede o **efeito causal** de uma campanha.
- Não diz que uma feature “causa” recompra.
- Não substitui teste A/B ou modelo de uplift.
- Não deve ser avaliado apenas por acurácia.

### Sinal de drift
Se PR-AUC/lift caem do período de validação para o teste, o comportamento dos clientes está mudando. Em produção, isso demanda monitoramento e retreino.

## 19. KPIs executivos do modelo

In [ ]:
best_val = validation_metrics.loc[validation_metrics['model'] == best_model_name].iloc[0]
best_test = test_metrics.iloc[0]

executive_kpis = pd.DataFrame({
    'metric': [
        'Clientes na base de modelagem',
        'Taxa geral de recompra 90d',
        'Modelo selecionado',
        'PR-AUC validação',
        'PR-AUC teste',
        'ROC-AUC teste',
        'Lift@10% validação',
        'Lift@10% teste',
        'Recall@10% teste',
        'Precision@10% teste'
    ],
    'value': [
        len(model_base),
        model_base['repurchase_90d'].mean(),
        best_model_name,
        best_val['pr_auc'],
        best_test['pr_auc'],
        best_test['roc_auc'],
        best_val['lift_at_10pct'],
        best_test['lift_at_10pct'],
        best_test['recall_at_10pct'],
        best_test['precision_at_10pct']
    ]
})
display(executive_kpis)

## 20. Export dos resultados e do pipeline

In [ ]:
all_metrics = pd.concat([validation_metrics, test_metrics], ignore_index=True)

exports = {
    'repurchase_model_metrics.csv': all_metrics,
    'repurchase_test_scored_customers.csv': scored_test,
    'repurchase_score_deciles.csv': decile_summary,
    'repurchase_campaign_simulation.csv': campaign_summary,
    'repurchase_feature_importance.csv': feature_importance,
    'repurchase_monthly_test_performance.csv': monthly_performance,
    'repurchase_model_executive_kpis.csv': executive_kpis,
    'repurchase_model_split_summary.csv': split_summary,
    'repurchase_model_calibration.csv': calibration_table
}

for filename, df in exports.items():
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False)
    print('Exportado:', path, df.shape)

model_file = MODEL_DIR / 'repurchase_90d_pipeline.joblib'
joblib.dump(final_model, model_file)
print('Modelo salvo:', model_file)

metadata = {
    'target': 'repurchase_90d',
    'horizon_days': 90,
    'selected_model': best_model_name,
    'feature_columns': feature_columns,
    'train_validation_end': str(train_valid_df['first_purchase'].max()),
    'test_start': str(test_df['first_purchase'].min()),
    'test_end': str(test_df['first_purchase'].max()),
    'random_state': RANDOM_STATE
}
metadata_file = MODEL_DIR / 'repurchase_90d_metadata.json'
metadata_file.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
print('Metadata salva:', metadata_file)

## 21. Sanity checks finais

In [ ]:
assert len(train_df) + len(valid_df) + len(test_df) == len(model_base)
assert train_df['first_purchase'].max() <= valid_df['first_purchase'].min()
assert valid_df['first_purchase'].max() <= test_df['first_purchase'].min()
assert scored_test['repurchase_score'].between(0, 1).all()
assert set(scored_test['score_decile'].unique()) == set(range(1, 11))
assert final_model is not None
assert (PROCESSED_DIR / 'repurchase_model_metrics.csv').exists()
assert (MODEL_DIR / 'repurchase_90d_pipeline.joblib').exists()

print('Sanity checks concluídos com sucesso.')

## 22. Conclusão para o relatório executivo

A previsão de recompra deve ser apresentada como um problema de **priorização**, não de classificação binária tradicional. Com uma taxa de recompra em 90 dias muito baixa, o ganho econômico vem de concentrar clientes mais promissores no topo do ranking.

A comparação out-of-time também permite uma conclusão de governança: **o desempenho preditivo precisa ser monitorado ao longo do tempo**, pois mudanças na composição de clientes, categorias, preços e aquisição podem reduzir o poder do modelo.

### Próximos passos recomendados
1. incorporar sinais adicionais da experiência da primeira compra (prazo/logística e review, disponíveis somente quando não violarem o momento de scoring desejado);
2. avaliar Gradient Boosting/XGBoost/LightGBM em benchmark controlado;
3. calibrar probabilidades se o score for usado como probabilidade literal;
4. implementar teste A/B para medir incremento real de campanhas;
5. evoluir para **uplift modeling**, separando clientes persuadíveis daqueles que recomprariam espontaneamente.